In [76]:
import pandas as pd
import os
from abc import ABC, abstractmethod
from typing import Union, List
from pydantic.dataclasses import dataclass

In [77]:
os.listdir('../data/raw/2024/')[:5]

['INMET_CO_DF_A001_BRASILIA_01-01-2024_A_31-12-2024.CSV',
 'INMET_CO_DF_A042_BRAZLANDIA_01-01-2024_A_31-12-2024.CSV',
 'INMET_CO_DF_A045_AGUAS EMENDADAS_01-01-2024_A_31-12-2024.CSV',
 'INMET_CO_DF_A046_GAMA (PONTE ALTA)_01-01-2024_A_31-12-2024.CSV',
 'INMET_CO_DF_A047_PARANOA (COOPA-DF)_01-01-2024_A_31-12-2024.CSV']

In [78]:
csv_path = '../data/raw/2024/INMET_NE_BA_A401_SALVADOR_01-01-2024_A_31-12-2024.CSV'
df = pd.read_csv(csv_path, sep=';', encoding='latin-1', skiprows=lambda x: x in range(8))
 
df.head()

,Data,Hora UTC,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (Kj/m²),"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",TEMPERATURA DO PONTO DE ORVALHO (°C),TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C),TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIREÇÃO HORARIA (gr) (° (gr))","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",Unnamed: 19
0,2024/01/01,0000 UTC,0,"1006,7","1006,7","1005,8",NaN,"26,8","22,8","26,8","26,6","22,8","22,6",79.0,78.0,79.0,66.0,"5,3","1,4",NaN
1,2024/01/01,0100 UTC,0,"1006,9","1006,9","1006,7",NaN,"26,7","22,6","26,8","26,5","22,8","22,4",79.0,77.0,78.0,62.0,"5,5","1,2",NaN
2,2024/01/01,0200 UTC,0,"1006,9","1007,1","1006,8",NaN,"26,5","22,9","26,7","26,4",23,"22,5",81.0,78.0,81.0,75.0,"4,6","1,2",NaN
3,2024/01/01,0300 UTC,0,"1006,5","1006,9","1006,5",NaN,"26,3","22,7","26,5","26,2","22,9","22,6",82.0,80.0,81.0,69.0,"4,6","1,1",NaN
4,2024/01/01,0400 UTC,0,"1006,5","1006,6","1006,5",NaN,26,"22,6","26,3","25,7","22,7","22,5",82.0,81.0,81.0,56.0,"4,2",",8",NaN


In [158]:
class IDataEngineering(ABC):

    @abstractmethod
    def _load_all_data(self):
        pass

    @abstractmethod
    def _default_read_csv(self):
        pass

    @abstractmethod
    def _cleaning_str_data_hours_columns(self):
        pass
    

In [ ]:
@dataclass
class DataEngInput:
    csv_paths: List[str]

class DataEngineering(IDataEngineering):
    def __init__(self, csv_paths:Union[str, List[str]]):

        paths = [csv_paths] if isinstance(csv_paths, str) else csv_paths
        self.input = DataEngInput(csv_paths=paths)

        self.raw_dataframes = self._load_all_data()
        self.process_data()

    def _default_read_csv(self,path:str) -> pd.DataFrame:
        _df = pd.read_csv(path, sep=';', encoding='latin-1', skiprows=lambda x: x in range(8))
        return _df.iloc[:,:-1]

    def _load_all_data(self) -> List[pd.DataFrame]:
        return [self._default_read_csv(path) for path in self.input.csv_paths]

    def _cleaning_str_data_hours_columns(self, df:pd.DataFrame) -> pd.DataFrame:
        df['DATA_HORA'] = df['Data'] + df['Hora UTC'].str.strip('UTC').str.strip(' ')
        df['DATA_HORA'] = pd.to_datetime(df['DATA_HORA'], format='%Y/%m/%d%H%M')
        return df.drop(['Data', 'Hora UTC'], axis=1)
    

    def _convert_str_to_numeric(self, df:pd.DataFrame, dtype:List[str]=['object', 'string']) -> pd.DataFrame:
        col_strings_type = df.select_dtypes(include=dtype).columns
        for col in col_strings_type:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.', regex=False), errors='coerce')
        return df
    
    def _rename_all_columns(self, df:pd.DataFrame) -> pd.DataFrame:
        rename_map = {
            'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'precipitacao_total_mm',
            'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)': 'pressao_atm_estacao_mb',
            'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)': 'pressao_atm_max_mb',
            'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)': 'pressao_atm_min_mb',
            'RADIACAO GLOBAL (Kj/m²)': 'radiacao_global_kj_m2',
            'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)': 'temp_ar_c',
            'TEMPERATURA DO PONTO DE ORVALHO (°C)': 'temp_ponto_orvalho_c',
            'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'temp_max_c',
            'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'temp_min_c',
            'TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)': 'temp_orvalho_max_c',
            'TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)': 'temp_orvalho_min_c',
            'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)': 'umidade_rel_max_percent',
            'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)': 'umidade_rel_min_percent',
            'UMIDADE RELATIVA DO AR, HORARIA (%)': 'umidade_rel_ar_percent',
            'VENTO, DIREÇÃO HORARIA (gr) (° (gr))': 'vento_direcao_graus',
            'VENTO, RAJADA MAXIMA (m/s)': 'vento_rajada_ms',
            'VENTO, VELOCIDADE HORARIA (m/s)': 'vento_vel_ms',
            'DATA_HORA': 'data_hora'
        }

        return df.rename(columns=rename_map)

    def process_data(self) -> List[pd.DataFrame]:
        """Aplica a transformação em toda a lista de dataframes."""
        self.raw_dataframes = [self._cleaning_str_data_hours_columns(df) for df in self.raw_dataframes]
        self.raw_dataframes = [self._convert_str_to_numeric(df) for df in self.raw_dataframes]
        self.raw_dataframes = [self._rename_all_columns(df) for df in self.raw_dataframes]
            


In [192]:
data_test = DataEngineering(csv_paths=['../data/raw/2024/INMET_NE_BA_A401_SALVADOR_01-01-2024_A_31-12-2024.CSV'])
data_test = data_test.raw_dataframes[0]

In [193]:
df_ = data_test.select_dtypes(include=['str']).columns
for col in df_:
    print(col)

In [ ]:
data_test.columns

Index(['precipitacao_total_mm', 'pressao_atm_estacao_mb', 'pressao_atm_max_mb',
       'pressao_atm_min_mb', 'radiacao_global_kj_m2', 'temp_ar_c',
       'temp_ponto_orvalho_c', 'temp_max_c', 'temp_min_c',
       'temp_orvalho_max_c', 'temp_orvalho_min_c', 'umidade_rel_max_percent',
       'umidade_rel_min_percent', 'umidade_rel_ar_percent',
       'vento_direcao_graus', 'vento_rajada_ms', 'vento_vel_ms', 'data_hora'],
      dtype='str')

In [ ]:
pd.to_datetime(data_test.raw_dataframes['Data'] + data_test.raw_dataframes['Hora UTC'].str.strip('UTC').str.strip(' '),
               format='%Y/%m/%d%H%M', 
               errors='coerce')

0      2024-01-01 00:00:00
1      2024-01-01 01:00:00
2      2024-01-01 02:00:00
3      2024-01-01 03:00:00
4      2024-01-01 04:00:00
               ...        
8779   2024-12-31 19:00:00
8780   2024-12-31 20:00:00
8781   2024-12-31 21:00:00
8782   2024-12-31 22:00:00
8783   2024-12-31 23:00:00
Length: 8784, dtype: datetime64[us]